In [8]:
import os
import cv2
import numpy as np

VIDEO_DIR = "./datasets/data-from-juniors/videos"
FRAME_DIR = "./datasets/data-from-juniors/frames"

NUM_FRAMES = 8
IMG_SIZE = 224

import os
import cv2
import torch
import numpy as np
import pandas as pd
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import f1_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

VIDEO_DIR = "./datasets/data-from-juniors/videos"
CSV_PATH = "./datasets/data-from-juniors/dataset.csv"

NUM_FRAMES = 8
IMG_SIZE = 224
BATCH_SIZE = 4
EPOCHS = 5
df = pd.read_csv(CSV_PATH)

rare_labels = [
    "ethinity_hate",
    "caste_based_hate",
    "social_hate",
    "religion_hate",
    "controversial"
]

df["rare_hate"] = df[rare_labels].max(axis=1)
df = df.drop(columns=rare_labels)


LABEL_COLUMNS = [
    'sensitive','derogatory__lang','threat','sexuality_hate',
    'nationality_hate','political_hate','anger','emotional',
    'indv_hate','gender_hate','rare_hate'
]

# keep only needed columns
df = df[["video_id"] + LABEL_COLUMNS]

# remove empty label rows
df["label_sum"] = df[LABEL_COLUMNS].sum(axis=1)
df = df[df["label_sum"] > 0]
df = df.drop(columns=["label_sum"])

df = df.reset_index(drop=True)

print("Final samples:", len(df))

Final samples: 1788


In [13]:
df

,video_id,sensitive,derogatory__lang,threat,sexuality_hate,nationality_hate,political_hate,anger,emotional,indv_hate,gender_hate,rare_hate
0,54,1,0,0,0,0,0,0,0,0,0,0
1,55,1,0,0,0,0,0,0,0,0,0,0
2,56,1,0,0,0,0,0,0,0,0,0,0
3,57,1,0,0,0,0,0,0,0,0,0,0
4,58,1,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1783,5146,0,1,0,0,0,0,0,0,0,0,0
1784,5147,1,1,0,0,0,0,0,0,0,0,1
1785,5149,1,1,0,0,0,0,0,0,0,0,0
1786,5152,0,0,0,0,0,0,0,0,0,0,1


In [9]:
import cv2
import numpy as np
import logging

NUM_FRAMES = 8
IMG_SIZE = 224

os.makedirs(FRAME_DIR, exist_ok=True)
logging.basicConfig(level=logging.INFO)


def extract_frames(video_path, save_dir):
    cap = cv2.VideoCapture(video_path)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        return False

    indices = np.linspace(0, total_frames - 1, NUM_FRAMES).astype(int)

    os.makedirs(save_dir, exist_ok=True)

    for i, idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        success, frame = cap.read()

        if not success:
            continue

        frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        cv2.imwrite(os.path.join(save_dir, f"{i}.jpg"), frame)

    cap.release()
    return True

In [14]:
video_map = {}

for f in os.listdir(VIDEO_DIR):
    name, ext = os.path.splitext(f)
    video_map[name] = f  # "1002" → "1002.mp4"

In [15]:
video_map

{'1': '1.mp4',
 '10': '10.mp4',
 '100': '100.mp4',
 '1000': '1000.webm',
 '1001': '1001.webm',
 '1002': '1002.mp4',
 '1003': '1003.webm',
 '1004': '1004.webm',
 '1005': '1005.webm',
 '1006': '1006.webm',
 '1007': '1007.webm',
 '1008': '1008.webm',
 '1009': '1009.webm',
 '101': '101.mp4',
 '1010': '1010.webm',
 '1011': '1011.webm',
 '1012': '1012.webm',
 '1013': '1013.webm',
 '1014': '1014.webm',
 '1015': '1015.webm',
 '1016': '1016.webm',
 '1017': '1017.webm',
 '1018': '1018.webm',
 '1019': '1019.webm',
 '102': '102.mp4',
 '1020': '1020.webm',
 '1021': '1021.webm',
 '1022': '1022.webm',
 '1023': '1023.webm',
 '1024': '1024.webm',
 '1025': '1025.webm',
 '1026': '1026.webm',
 '1027': '1027.webm',
 '1028': '1028.webm',
 '1029': '1029.webm',
 '103': '103.mp4',
 '1030': '1030.webm',
 '1031': '1031.webm',
 '1032': '1032.webm',
 '1033': '1033.webm',
 '1034': '1034.webm',
 '1035': '1035.webm',
 '1036': '1036.webm',
 '1037': '1037.webm',
 '1038': '1038.webm',
 '1039': '1039.webm',
 '104': '104.

In [17]:
for i, row in df.iterrows():
    # video_file = str(row["video_id"])
    video_id = str(row["video_id"])

    if video_id not in video_map:
        print(f"Missing video: {video_id}")
        continue

    video_file = video_map[video_id]
    video_path = os.path.join(VIDEO_DIR, video_file)

    video_path = os.path.join(VIDEO_DIR, str(video_file))
    clip_id = os.path.splitext(video_file)[0]
    save_dir = os.path.join(FRAME_DIR, clip_id)

    if os.path.exists(save_dir):
        logging.info(f"SKIP {video_file}")
        continue

    success = extract_frames(video_path, save_dir)

    if success:
        logging.info(f"[{i}/{len(df)}] DONE {video_file}")
    else:
        logging.warning(f"[{i}] FAILED {video_file}")

INFO:root:SKIP 54.mp4
INFO:root:[1/1788] DONE 55.mp4
INFO:root:[2/1788] DONE 56.mp4
INFO:root:[3/1788] DONE 57.mp4
INFO:root:[4/1788] DONE 58.mp4
INFO:root:[5/1788] DONE 59.mp4
INFO:root:[6/1788] DONE 60.mp4
INFO:root:[7/1788] DONE 61.mp4
INFO:root:[8/1788] DONE 62.mp4
INFO:root:[9/1788] DONE 63.mp4
INFO:root:[10/1788] DONE 64.mp4
INFO:root:[11/1788] DONE 65.mp4
INFO:root:[12/1788] DONE 67.mp4
INFO:root:[13/1788] DONE 68.mp4
INFO:root:[14/1788] DONE 69.mp4
INFO:root:[15/1788] DONE 77.mp4
INFO:root:[16/1788] DONE 80.mp4
INFO:root:[17/1788] DONE 81.mp4
INFO:root:[18/1788] DONE 88.mp4
INFO:root:[19/1788] DONE 89.mp4
INFO:root:[20/1788] DONE 90.mp4
INFO:root:[21/1788] DONE 91.mp4
INFO:root:[22/1788] DONE 93.mp4
INFO:root:[23/1788] DONE 94.mp4
INFO:root:[24/1788] DONE 95.mp4
INFO:root:[25/1788] DONE 96.mp4
INFO:root:[26/1788] DONE 97.mp4
INFO:root:[27/1788] DONE 98.mp4
INFO:root:[28/1788] DONE 99.mp4
INFO:root:[29/1788] DONE 100.mp4
INFO:root:[30/1788] DONE 101.mp4
INFO:root:[31/1788] DONE 

Missing video: 132


INFO:root:[64/1788] DONE 135.mp4
INFO:root:[65/1788] DONE 138.mp4
INFO:root:[66/1788] DONE 139.mp4
INFO:root:[67/1788] DONE 143.mp4
INFO:root:[68/1788] DONE 145.mp4
INFO:root:[69/1788] DONE 147.mp4
INFO:root:[70/1788] DONE 155.mp4
INFO:root:[71/1788] DONE 156.mp4
INFO:root:[72/1788] DONE 157.mp4
INFO:root:[73/1788] DONE 158.mp4
INFO:root:[74/1788] DONE 162.mp4
INFO:root:[75/1788] DONE 163.mp4
INFO:root:[76/1788] DONE 166.mp4
INFO:root:[77/1788] DONE 170.mp4
INFO:root:[78/1788] DONE 171.mp4
INFO:root:[79/1788] DONE 172.mp4
INFO:root:[80/1788] DONE 175.mp4
INFO:root:[81/1788] DONE 180.mp4
INFO:root:[82/1788] DONE 181.mp4
INFO:root:[83/1788] DONE 182.mp4
INFO:root:[84/1788] DONE 187.mp4
INFO:root:[85/1788] DONE 188.mp4
INFO:root:[86/1788] DONE 189.mp4
INFO:root:[87/1788] DONE 191.mp4
INFO:root:[88/1788] DONE 196.mp4
INFO:root:[89/1788] DONE 198.mp4
INFO:root:[90/1788] DONE 199.mp4
INFO:root:[91/1788] DONE 200.mp4
INFO:root:[92/1788] DONE 201.mp4
INFO:root:[93/1788] DONE 204.mp4
INFO:root:

Missing video: 576


INFO:root:[249/1788] DONE 586.mp4
INFO:root:[250/1788] DONE 588.mp4
INFO:root:[251/1788] DONE 591.mp4
INFO:root:[252/1788] DONE 594.mp4
INFO:root:[253/1788] DONE 596.mp4
INFO:root:[254/1788] DONE 597.mp4
INFO:root:[255/1788] DONE 598.mp4
INFO:root:[256/1788] DONE 599.mp4
INFO:root:[257/1788] DONE 600.mp4
INFO:root:[258/1788] DONE 602.mp4
INFO:root:[259/1788] DONE 604.mp4
INFO:root:[260/1788] DONE 609.mp4
INFO:root:[261/1788] DONE 610.mp4
INFO:root:[262/1788] DONE 612.mp4
INFO:root:[263/1788] DONE 613.mp4
INFO:root:[264/1788] DONE 614.mp4
INFO:root:[265/1788] DONE 621.mp4
INFO:root:[266/1788] DONE 626.mp4
INFO:root:[267/1788] DONE 628.mp4
INFO:root:[268/1788] DONE 633.mp4
INFO:root:[269/1788] DONE 634.mp4
INFO:root:[270/1788] DONE 643.mp4
INFO:root:[271/1788] DONE 646.mp4
INFO:root:[272/1788] DONE 648.mp4
INFO:root:[273/1788] DONE 649.mp4
INFO:root:[274/1788] DONE 653.mp4
INFO:root:[275/1788] DONE 654.mp4
INFO:root:[276/1788] DONE 658.mp4
INFO:root:[277/1788] DONE 663.mp4
INFO:root:[278

Missing video: 2901


INFO:root:[958/1788] DONE 2903.webm
INFO:root:[959/1788] DONE 2904.webm
INFO:root:[960/1788] DONE 2907.webm
INFO:root:[961/1788] DONE 2921.webm
INFO:root:[962/1788] DONE 2924.webm
INFO:root:[963/1788] DONE 2935.webm
INFO:root:[964/1788] DONE 2944.webm
INFO:root:[965/1788] DONE 2953.webm
INFO:root:[966/1788] DONE 2956.webm
INFO:root:[967/1788] DONE 2960.webm
INFO:root:[968/1788] DONE 2967.webm
INFO:root:[969/1788] DONE 2969.webm
INFO:root:[970/1788] DONE 2990.webm
INFO:root:[971/1788] DONE 2995.webm
INFO:root:[972/1788] DONE 3000.mp4
INFO:root:[973/1788] DONE 3001.webm
INFO:root:[974/1788] DONE 3002.webm
INFO:root:[975/1788] DONE 3003.webm
INFO:root:[976/1788] DONE 3004.webm
INFO:root:[977/1788] DONE 3005.webm
INFO:root:[978/1788] DONE 3011.webm
INFO:root:[979/1788] DONE 3012.webm
INFO:root:[980/1788] DONE 3014.webm
INFO:root:[981/1788] DONE 3016.webm
INFO:root:[982/1788] DONE 3017.webm
INFO:root:[983/1788] DONE 3022.webm
INFO:root:[984/1788] DONE 3024.webm
INFO:root:[985/1788] DONE 302

Missing video: 4527


INFO:root:[1516/1788] DONE 4530.mp4
INFO:root:[1517/1788] DONE 4531.mp4
INFO:root:[1518/1788] DONE 4532.mp4
INFO:root:[1519/1788] DONE 4536.mp4
INFO:root:[1520/1788] DONE 4537.mp4
INFO:root:[1521/1788] DONE 4542.mp4
INFO:root:[1522/1788] DONE 4543.mp4
INFO:root:[1523/1788] DONE 4545.mp4
INFO:root:[1524/1788] DONE 4547.mp4
INFO:root:[1525/1788] DONE 4549.mp4
INFO:root:[1526/1788] DONE 4551.mp4
INFO:root:[1527/1788] DONE 4552.mp4
INFO:root:[1528/1788] DONE 4555.mp4
INFO:root:[1529/1788] DONE 4560.mp4
INFO:root:[1530/1788] DONE 4561.mp4
INFO:root:[1531/1788] DONE 4563.mp4
INFO:root:[1532/1788] DONE 4565.mp4
INFO:root:[1533/1788] DONE 4572.mp4
INFO:root:[1534/1788] DONE 4576.mp4
INFO:root:[1535/1788] DONE 4581.mp4
INFO:root:[1536/1788] DONE 4583.mp4
INFO:root:[1537/1788] DONE 4584.mp4
INFO:root:[1538/1788] DONE 4586.mp4
INFO:root:[1539/1788] DONE 4587.mp4
INFO:root:[1540/1788] DONE 4589.mp4
INFO:root:[1541/1788] DONE 4590.mp4
INFO:root:[1542/1788] DONE 4592.mp4
INFO:root:[1543/1788] DONE 4